In [1]:
import torch
import clip
from collections import OrderedDict

# Load base CLIP model
device = "cuda" if torch.cuda.is_available() else "cpu"
model, _ = clip.load("ViT-B/16", device=device)

# Load LoRA weights
lora_path = "/home/sunayana/Documents/Concept_LoRA/lora_weights/vitb16/ucf101/16shots/seed1/lora_weights.pt"
lora_state = torch.load(lora_path, map_location=device)

# If saved as state_dict only
if "state_dict" in lora_state:
    lora_state = lora_state["state_dict"]

print(f"Loaded LoRA state with {len(lora_state)} tensors.")


Loaded LoRA state with 2 tensors.


In [2]:
for k, v in lora_state.items():
    print(f"\nTop-level key: {k}")
    if isinstance(v, dict):
        for subk, subv in v.items():
            if torch.is_tensor(subv):
                print(f"  {k}.{subk}: {tuple(subv.shape)}, dtype={subv.dtype}")
            else:
                print(f"  {k}.{subk}: {type(subv)}")
    else:
        print(f"  {type(v)}")



Top-level key: weights
  weights.layer_0: <class 'dict'>
  weights.layer_1: <class 'dict'>
  weights.layer_2: <class 'dict'>
  weights.layer_3: <class 'dict'>
  weights.layer_4: <class 'dict'>
  weights.layer_5: <class 'dict'>
  weights.layer_6: <class 'dict'>
  weights.layer_7: <class 'dict'>
  weights.layer_8: <class 'dict'>
  weights.layer_9: <class 'dict'>
  weights.layer_10: <class 'dict'>
  weights.layer_11: <class 'dict'>
  weights.layer_12: <class 'dict'>
  weights.layer_13: <class 'dict'>
  weights.layer_14: <class 'dict'>
  weights.layer_15: <class 'dict'>
  weights.layer_16: <class 'dict'>
  weights.layer_17: <class 'dict'>
  weights.layer_18: <class 'dict'>
  weights.layer_19: <class 'dict'>
  weights.layer_20: <class 'dict'>
  weights.layer_21: <class 'dict'>
  weights.layer_22: <class 'dict'>
  weights.layer_23: <class 'dict'>

Top-level key: metadata
  metadata.r: <class 'int'>
  metadata.alpha: <class 'int'>
  metadata.encoder: <class 'str'>
  metadata.params: <class '

In [4]:
for lname, ldict in lora_state["weights"].items():
    print(f"\nLayer: {lname}")
    for subk, subv in ldict.items():
        if isinstance(subv, dict):
            for kk, vv in subv.items():
                if torch.is_tensor(vv):
                    print(f"  {lname}.{subk}.{kk}: {tuple(vv.shape)}")
        elif torch.is_tensor(subv):
            print(f"  {lname}.{subk}: {tuple(subv.shape)}")



Layer: layer_0
  layer_0.q_proj.w_lora_A: (2, 512)
  layer_0.q_proj.w_lora_B: (512, 2)
  layer_0.k_proj.w_lora_A: (2, 512)
  layer_0.k_proj.w_lora_B: (512, 2)
  layer_0.v_proj.w_lora_A: (2, 512)
  layer_0.v_proj.w_lora_B: (512, 2)

Layer: layer_1
  layer_1.q_proj.w_lora_A: (2, 512)
  layer_1.q_proj.w_lora_B: (512, 2)
  layer_1.k_proj.w_lora_A: (2, 512)
  layer_1.k_proj.w_lora_B: (512, 2)
  layer_1.v_proj.w_lora_A: (2, 512)
  layer_1.v_proj.w_lora_B: (512, 2)

Layer: layer_2
  layer_2.q_proj.w_lora_A: (2, 512)
  layer_2.q_proj.w_lora_B: (512, 2)
  layer_2.k_proj.w_lora_A: (2, 512)
  layer_2.k_proj.w_lora_B: (512, 2)
  layer_2.v_proj.w_lora_A: (2, 512)
  layer_2.v_proj.w_lora_B: (512, 2)

Layer: layer_3
  layer_3.q_proj.w_lora_A: (2, 512)
  layer_3.q_proj.w_lora_B: (512, 2)
  layer_3.k_proj.w_lora_A: (2, 512)
  layer_3.k_proj.w_lora_B: (512, 2)
  layer_3.v_proj.w_lora_A: (2, 512)
  layer_3.v_proj.w_lora_B: (512, 2)

Layer: layer_4
  layer_4.q_proj.w_lora_A: (2, 512)
  layer_4.q_proj.w_l

In [6]:
import torch
import clip
from pathlib import Path

# ============================================================
# 1. Load base CLIP and LoRA checkpoint
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"

model, _ = clip.load("ViT-B/16", device=device)
print("Base CLIP loaded.")

lora_path = Path("/home/sunayana/Documents/Concept_LoRA/lora_weights/vitb16/ucf101/16shots/seed1/lora_weights.pt")
lora_state = torch.load(lora_path, map_location=device)
layers = lora_state["weights"]
meta = lora_state["metadata"]

r, alpha = meta["r"], meta["alpha"]
scale = alpha / r
print(f"LoRA rank={r}, alpha={alpha}, scale={scale}, layers={len(layers)}")

# ============================================================
# 2. Debug: Check actual structure
# ============================================================
print("\n=== Checking LoRA structure ===")
layer_0 = layers["layer_0"]
print(f"layer_0 keys: {layer_0.keys()}")
print(f"layer_0 type: {type(layer_0)}")

# Check if nested
if "q_proj" in layer_0 and isinstance(layer_0["q_proj"], dict):
    print("Structure is nested dict")
    print(f"layer_0['q_proj'] keys: {layer_0['q_proj'].keys()}")
else:
    print("Structure is flat")
    for k in list(layer_0.keys())[:5]:
        print(f"  {k}: {layer_0[k].shape if torch.is_tensor(layer_0[k]) else type(layer_0[k])}")

# ============================================================
# 3. Fixed merge helper function
# ============================================================
def apply_lora_to_block(block, lora_dict, scale):
    """
    Merges LoRA weights into q, k, v projection layers of a transformer block.
    block: CLIP transformer block
    lora_dict: dict of LoRA matrices for that block (A/B pairs)
    scale: scaling factor (alpha / r)
    """
    for proj_name in ["q_proj", "k_proj", "v_proj"]:
        # Handle nested dict structure: layer_0 -> q_proj -> w_lora_A
        if proj_name in lora_dict and isinstance(lora_dict[proj_name], dict):
            # Nested structure
            A = lora_dict[proj_name]["w_lora_A"].to(block.attn.in_proj_weight.device)
            B = lora_dict[proj_name]["w_lora_B"].to(block.attn.in_proj_weight.device)
        else:
            # Flat structure: try direct keys
            try:
                A = lora_dict[f"{proj_name}.w_lora_A"].to(block.attn.in_proj_weight.device)
                B = lora_dict[f"{proj_name}.w_lora_B"].to(block.attn.in_proj_weight.device)
            except KeyError:
                print(f"Skipping {proj_name} — not found in LoRA dict")
                continue

        # Convert LoRA weight shapes (r, d) and (d, r) to (d, d)
        # NOTE: Check if you need to transpose - depends on LoRA implementation
        # LoRA update: W' = W + BA * scale
        delta_w = scale * (B @ A)

        # Add LoRA update to base projection weight
        # CLIP uses combined in_proj_weight for q, k, v
        if hasattr(block.attn, "in_proj_weight"):  
            w = block.attn.in_proj_weight.data
            d_model = w.shape[1]  # 512 for ViT-B/16 text, 768 for vision
            
            # The combined weight is [3*d_model, d_model]
            # Split into q, k, v sections
            if proj_name == "q_proj":
                w[:d_model, :] += delta_w
            elif proj_name == "k_proj":
                w[d_model:2*d_model, :] += delta_w
            elif proj_name == "v_proj":
                w[2*d_model:, :] += delta_w
                
            print(f"  ✓ Applied LoRA to {proj_name}: delta_w shape={delta_w.shape}, max_change={delta_w.abs().max():.6f}")
        else:
            print(f"Could not locate in_proj_weight in block.")


# ============================================================
# 4. Apply LoRA updates to both encoders
# ============================================================

print("\n=== Merging Text Encoder ===")
# Text encoder: layers 0–11
for i in range(12):
    lora_dict = layers[f"layer_{i}"]
    print(f"Layer {i}:")
    apply_lora_to_block(model.transformer.resblocks[i], lora_dict, scale)

print("\n=== Merging Vision Encoder ===")
# Vision encoder: layers 12–23
for i in range(12, 24):
    lora_dict = layers[f"layer_{i}"]
    print(f"Layer {i} (vision {i-12}):")
    apply_lora_to_block(model.visual.transformer.resblocks[i - 12], lora_dict, scale)


# ============================================================
# 5. Verify merge worked
# ============================================================
print("\n=== Verifying Merge ===")
base_model, _ = clip.load("ViT-B/16", device=device)

# Check text encoder
text_diff = (model.transformer.resblocks[0].attn.in_proj_weight - 
             base_model.transformer.resblocks[0].attn.in_proj_weight).abs().max().item()
print(f"Text encoder layer 0 max diff: {text_diff:.8f}")

# Check vision encoder  
vision_diff = (model.visual.transformer.resblocks[0].attn.in_proj_weight - 
               base_model.visual.transformer.resblocks[0].attn.in_proj_weight).abs().max().item()
print(f"Vision encoder layer 0 max diff: {vision_diff:.8f}")

if text_diff > 0 or vision_diff > 0:
    print("✓ LoRA weights successfully merged!")
else:
    print("✗ WARNING: No changes detected - merge may have failed")


# ============================================================
# 6. Save merged model
# ============================================================
save_path = Path("clip_vitb16_ucf101_lora_merged.pt")
torch.save(model.state_dict(), save_path)
print(f"\nMerged model saved to {save_path.resolve()}")

Base CLIP loaded.
LoRA rank=2, alpha=1, scale=0.5, layers=24

=== Checking LoRA structure ===
layer_0 keys: dict_keys(['q_proj', 'k_proj', 'v_proj'])
layer_0 type: <class 'dict'>
Structure is nested dict
layer_0['q_proj'] keys: dict_keys(['w_lora_A', 'w_lora_B'])

=== Merging Text Encoder ===
Layer 0:
  ✓ Applied LoRA to q_proj: delta_w shape=torch.Size([512, 512]), max_change=0.001824
  ✓ Applied LoRA to k_proj: delta_w shape=torch.Size([512, 512]), max_change=0.001801
  ✓ Applied LoRA to v_proj: delta_w shape=torch.Size([512, 512]), max_change=0.001772
Layer 1:
  ✓ Applied LoRA to q_proj: delta_w shape=torch.Size([512, 512]), max_change=0.002019
  ✓ Applied LoRA to k_proj: delta_w shape=torch.Size([512, 512]), max_change=0.002303
  ✓ Applied LoRA to v_proj: delta_w shape=torch.Size([512, 512]), max_change=0.002757
Layer 2:
  ✓ Applied LoRA to q_proj: delta_w shape=torch.Size([512, 512]), max_change=0.001859
  ✓ Applied LoRA to k_proj: delta_w shape=torch.Size([512, 512]), max_change=

In [7]:
# ...existing code...

import torch

ckpt_path = "/home/sunayana/Documents/Concept_LoRA/clip_vitb16_pets_lora_merged.pt"
ckpt = torch.load(ckpt_path, map_location="cpu")

# Try to find the actual model state dict inside the checkpoint
candidate_keys = ["model_state_dict", "state_dict", "model"]
state_dict = None
for k in candidate_keys:
    if k in ckpt and isinstance(ckpt[k], dict):
        state_dict = ckpt[k]
        print(f"Using nested key '{k}' as model state_dict")
        break

if state_dict is None:
    state_dict = ckpt
    print("No nested model state_dict key found; treating entire checkpoint as state_dict")

print(f"\nNumber of tensors in state_dict: {len(state_dict)}\n")

for name, tensor in state_dict.items():
    if torch.is_tensor(tensor):
        print(f"{name}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}")
        # Print a few example weights
        flat = tensor.view(-1)
        print("  sample values:", flat[:5])
    else:
        print(f"{name}: type={type(tensor)}")
# ...existing code...

No nested model state_dict key found; treating entire checkpoint as state_dict

Number of tensors in state_dict: 302

positional_embedding: shape=(77, 512), dtype=torch.float32
  sample values: tensor([-2.1003e-04, -8.2927e-05, -4.0602e-03, -4.9999e-03,  4.5176e-03])
text_projection: shape=(512, 512), dtype=torch.float16
  sample values: tensor([-0.0087,  0.0040, -0.0137, -0.0008,  0.0015], dtype=torch.float16)
logit_scale: shape=(), dtype=torch.float32
  sample values: tensor([4.6052])
visual.class_embedding: shape=(768,), dtype=torch.float32
  sample values: tensor([ 0.0172, -0.5701,  0.1725, -0.0019,  1.0032])
visual.positional_embedding: shape=(197, 768), dtype=torch.float32
  sample values: tensor([ 0.0001, -0.0420,  0.0619, -0.0024,  0.0841])
visual.proj: shape=(768, 512), dtype=torch.float16
  sample values: tensor([-0.0146, -0.0039, -0.0066, -0.0099, -0.0141], dtype=torch.float16)
visual.conv1.weight: shape=(768, 3, 16, 16), dtype=torch.float16
  sample values: tensor([-0.0095,

In [2]:
import torch

ckpt_path="/home/sunayana/Documents/Concept_LoRA/Discover-then-Name/lora_and_cubclf.pth"

ckpt = torch.load(ckpt_path, map_location="cpu")
for k, v in ckpt.items():
    if torch.is_tensor(v):
        print(f"{k}: {tuple(v.shape)}, dtype={v.dtype}")
    else:
        print(f"{k}: {type(v)}")

classifier.0.weight: (256, 512), dtype=torch.float32
classifier.0.bias: (256,), dtype=torch.float32
classifier.2.weight: (200, 256), dtype=torch.float32
classifier.2.bias: (200,), dtype=torch.float32
